# Apache Polaris — Rotate Principal

## Concept
Rotates a principal's OAuth credentials **in place** — unlike
`polaris_reset_principal.ipynb`, nothing is deleted. The principal, its
principal-role assignment, and its catalog-roles all survive untouched;
only the `clientId`/`clientSecret` pair changes.

## Why not just delete + recreate?
`polaris_reset_principal.ipynb` gets you a "clean" principal, but you pay
for it: the principal-role assignment and every catalog-role on its catalog
are gone too, and you have to rebuild the whole footprint by hand. Rotation
avoids that entirely — the credentials change, nothing else does.

## Two rotation APIs, two very different authorization stories
Polaris actually has **two** credential-rotation endpoints, and this
notebook deliberately runs into the first one's wall before using the one
that works for an admin:

1. **`POST /principals/{name}/rotate`** — **self-service only.** The caller
   must already be authenticated *as* the target principal. `root` /
   `service_admin` gets a **403** here even with full admin grants — this
   is by design, confirmed by a Polaris maintainer on
   [issue #624](https://github.com/apache/polaris/issues/624): *"users can
   rotate their own credentials but root cannot change them."*
2. **`POST /principals/{name}/reset`** — **admin-callable**, added in
   Polaris 1.2+ (present in this repo's 1.3.0 target) as the actual fix for
   #624. `root`/`service_admin` can reset *another* principal's credentials
   without deleting it. Gated by the `ENABLE_CREDENTIAL_RESET` server
   feature flag (default `true`), **not** by a grantable privilege.

## Why root can't just be granted the missing privilege
Polaris's RBAC model only has five securable objects: Catalog, Namespace,
Table, View, Policy. **Principal is not one of them.** There is no
`PRINCIPAL_ROTATE_CREDENTIALS`-style privilege string to grant to any role
— attempting to `PUT` such a grant 404s ("Unable to find matching target
resource method"), it doesn't 403. This notebook reproduces both the 403
and the 404 as documented negative results (Steps 2–3) before moving on to
the working `/reset` flow (Step 5).

## What about impersonation / token exchange?
Investigated and ruled out (Step 4): Polaris's OAuth endpoint only
implements the `client_credentials` grant — there is no token-exchange /
"act-as" grant type that would let root mint a token that *is* another
principal. Even if there were, root has no way to obtain it: a principal's
`clientSecret` is shown exactly once (at creation or rotation) and is never
retrievable again, so root has no credential material to impersonate with.
This is the same write-once-secret model that makes `/rotate` self-only in
the first place — the two restrictions reinforce each other by design.

## Flow
```
1. GET  Principal + Principal Role + Catalog Roles      (baseline snapshot)
2. POST /principals/{name}/rotate as root                → 403 (documented)
3. PUT  grant PRINCIPAL_ROTATE_CREDENTIALS to service_admin → 404 (documented)
4. Impersonation feasibility check                        → infeasible (documented)
5. POST /principals/{name}/reset as root                  → 200, new creds
6. VERIFY role wiring (principal-role, catalog-roles) is unchanged
7. POST /principals/{name}/rotate as the principal itself → 200, new creds
8. VERIFY the self-rotated (Step 7) credentials work — AND that the Step 5
   credentials still work too (see below — this is expected)
9. Rotate ONCE MORE to prove the Step 5 credentials finally die
```

## ⚠️ Credentials don't die after one rotation — Polaris keeps two secret slots
Measured on a live run (2026-07-08), not assumed: Polaris stores a **main**
and a **secondary** secret per principal (`PolarisPrincipalSecrets`, which
accepts either hash as valid). A plain `/rotate` only demotes the current
main secret to secondary and mints a new main — the demoted secret **keeps
working**. It takes a **second** rotation to push a given secret out of both
slots. `/reset`, by contrast, forces two internal rotations in one call, so
it fully evicts whatever secret was active right before it. Step 8 asserts
the Step 5 credentials are still alive (expected); Step 9 rotates again and
confirms they're finally dead. Don't assert immediate invalidation after a
single rotation — assert it after two.

Built on `src/polaris_rest.PolarisREST` (adds `reset_principal_credentials`
and `rotate_credentials` — see that module's docstrings for the full
authorization writeup).

## Safety — LOCAL DEVICE ONLY
Same rationale as every other notebook in `admin/`: hardcoded to the local
OrbStack host, with a host-guard assert, never wired to `init_env()` /
shared config. This notebook only rotates credentials (it doesn't delete
anything), but the target host guard still applies.

**Supersedes `polaris_rotate_credential.ipynb`**, which stopped at the
`/rotate` 403 without a working admin path — see the deprecation notice at
the top of that notebook.

## How to run
1. Set `PRINCIPAL_NAME` in the parameters cell.
2. Restart & Run All. New credentials print once per rotation step — save
   them if you need to actually use the principal afterward.


In [1]:
# ============================================================
# Prerequisites:  uv add requests
# ============================================================
import subprocess, sys, os

REQUIRED = ['requests']

def install(pkg):
    result = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)

⚠️  Restart kernel after first install.


In [2]:
# ============================================================
# Setup — hardcoded local device only (see Safety note above)
# ============================================================
import sys, pathlib
from IPython.display import display

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from polaris_rest import PolarisREST

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'
ROOT        = 'root'
ROOT_SECRET = 'polaris-secret'

# ── SAFETY GUARD: LOCAL DEVICE ONLY ──────────────────────────────────
assert any(t in POLARIS_URL for t in ("127.0.0.1", "localhost", "192.168.139.2")), \
    f"rotate_principal is LOCAL ONLY — refusing non-local host: {POLARIS_URL}"

pc = PolarisREST(POLARIS_URL, REALM)
r = pc.get_token(ROOT, ROOT_SECRET)
pc.token = r.json()['access_token']
print('✅ Connected as root')

✅ Connected as root


In [3]:
# ============================================================
# Parameters — set the target principal before running
# ============================================================
PRINCIPAL_NAME = 'REPLACE_ME'   # e.g. 'alice'

assert PRINCIPAL_NAME != 'REPLACE_ME', \
    'Set PRINCIPAL_NAME to a real principal before running this notebook.'

PRINCIPAL_ROLE = f'{PRINCIPAL_NAME}_role'

print(f'PRINCIPAL_NAME  = {PRINCIPAL_NAME}')
print(f'PRINCIPAL_ROLE  = {PRINCIPAL_ROLE}')

AssertionError: Set PRINCIPAL_NAME to a real principal before running this notebook.

In [ ]:
# ============================================================
# Step 1 — Baseline snapshot (principal + role wiring BEFORE rotation)
# ============================================================
r = pc.get_principal(PRINCIPAL_NAME)
assert r.status_code == 200, f'Principal not found ({r.status_code}): {PRINCIPAL_NAME}'
principal_before = r.json()
print(f'✅ Principal found: {PRINCIPAL_NAME}')
display(principal_before)

r = pc.get_principal_role(PRINCIPAL_ROLE)
principal_role_before = r.json() if r.status_code == 200 else None
print(f"{'✅' if principal_role_before else '⚠️ '} Principal role: {PRINCIPAL_ROLE} ({r.status_code})")

r = pc.list_principals_for_principal_role(PRINCIPAL_ROLE)
holders_before = sorted(p['name'] for p in r.json().get('principals', [])) if r.status_code == 200 else []
print(f'Principals holding {PRINCIPAL_ROLE}: {holders_before}')

In [ ]:
# ============================================================
# Step 2 — NEGATIVE RESULT: root attempts the self-service /rotate
# ============================================================
r = pc.rotate_credentials(PRINCIPAL_NAME)  # uses pc.token = root
print(f'Status: {r.status_code}')
print(r.text)

assert r.status_code == 403, \
    f'Expected 403 (self-only by design) — got {r.status_code}. Has upstream behavior changed?'
print("\n✅ Confirmed: /rotate is self-service only. root cannot use it on another principal.")

In [ ]:
# ============================================================
# Step 3 — NEGATIVE RESULT: attempt to grant the missing privilege
# ============================================================
# Principal is not a securable object in Polaris's RBAC model, so there is
# no PRINCIPAL_ROTATE_CREDENTIALS privilege to grant. This PUT targets the
# principal-roles grants endpoint the same way a real catalog-role grant
# would, to confirm it 404s rather than merely 403s.
import requests

r = requests.put(
    f'{pc.base_mgmt}/principal-roles/service_admin/grants',
    headers=pc._h(),
    json={'grant': {'type': 'principal', 'privilege': 'PRINCIPAL_ROTATE_CREDENTIALS'}},
)
print(f'Status: {r.status_code}')
print(r.text)

assert r.status_code == 404, \
    f'Expected 404 (no such grantable resource) — got {r.status_code}.'
print("\n✅ Confirmed: there is no privilege that closes this gap — not a missing grant, a missing securable object.")

In [ ]:
# ============================================================
# Step 4 — Impersonation feasibility check (documented, not executable)
# ============================================================
# Idea considered: have root obtain a token "as" PRINCIPAL_NAME (via an
# OAuth token-exchange / impersonation grant), then call the self-service
# /rotate with THAT token. Ruled out on two independent grounds:
#
#   1. Polaris's OAuth endpoint (/api/catalog/v1/oauth/tokens) only
#      implements the `client_credentials` grant. There is no
#      `urn:ietf:params:oauth:grant-type:token-exchange` (RFC 8693) or any
#      other "act-as" grant type registered (checked against
#      polaris.authentication.* config: authenticator/token-service/
#      token-broker are pluggable, but none expose an impersonation flow).
#   2. Even if such a grant type existed, root has no credential material
#      to use it with: a principal's clientSecret is returned ONCE, at
#      creation or rotation, and is never retrievable afterward (confirmed
#      by principal_before/principal_role_before above containing no
#      secret field). Root cannot authenticate "as" PRINCIPAL_NAME without
#      already possessing a secret it fundamentally cannot read.
#
# This is not a workaround waiting to be found — it's the same write-once
# -secret design that makes /rotate self-only in Step 2. The admin-callable
# /reset endpoint (Step 5) is Polaris's actual answer to this gap.
print('ℹ️  Impersonation ruled out: no token-exchange grant type exists, and root')
print('   cannot read an existing principal secret to fake one. See cell above for detail.')

In [ ]:
# ============================================================
# Step 5 — Root resets PRINCIPAL_NAME's credentials via the admin API
# ============================================================
r = pc.reset_principal_credentials(PRINCIPAL_NAME)
print(f'Status: {r.status_code}')

assert r.status_code == 200, (
    f'Reset failed ({r.status_code}): {r.text}\n'
    "If this 404s, ENABLE_CREDENTIAL_RESET may be disabled on this server."
)

creds = r.json().get('credentials', {})
CLIENT_ID     = creds.get('clientId')
CLIENT_SECRET = creds.get('clientSecret')
print(f'✅ New credentials for {PRINCIPAL_NAME} (root-issued reset):')
print(f'   clientId:     {CLIENT_ID}')
print(f'   clientSecret: {CLIENT_SECRET}')
print('\n⚠️  Save these — clientSecret is shown only once!')

In [ ]:
# ============================================================
# Step 6 — VERIFY role wiring survived (the whole point of rotating, not deleting)
# ============================================================
r = pc.get_principal(PRINCIPAL_NAME)
assert r.status_code == 200, f'Principal disappeared after reset?! ({r.status_code})'
print(f'✅ Principal still exists: {PRINCIPAL_NAME}')

r = pc.get_principal_role(PRINCIPAL_ROLE)
principal_role_after = r.json() if r.status_code == 200 else None
assert bool(principal_role_after) == bool(principal_role_before), \
    'Principal-role existence changed across rotation — unexpected side effect.'
print(f'✅ Principal role unchanged: {PRINCIPAL_ROLE}')

r = pc.list_principals_for_principal_role(PRINCIPAL_ROLE)
holders_after = sorted(p['name'] for p in r.json().get('principals', [])) if r.status_code == 200 else []
assert holders_after == holders_before, \
    f'Role assignment changed: before={holders_before} after={holders_after}'
print(f'✅ Role assignment unchanged: {holders_after}')
print('\n✅ Rotation preserved every bit of role wiring — unlike polaris_reset_principal.ipynb.')

In [ ]:
# ============================================================
# Step 7 — Self-rotate demo: PRINCIPAL_NAME rotates its OWN new credentials
# ============================================================
r = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{PRINCIPAL_ROLE}')
assert r.status_code == 200, f'Could not authenticate as {PRINCIPAL_NAME} with reset creds: {r.text}'
principal_token = r.json()['access_token']
print(f'✅ Authenticated as {PRINCIPAL_NAME} using the reset credentials')

r = pc.rotate_credentials(PRINCIPAL_NAME, token=principal_token)
print(f'Status: {r.status_code}')
assert r.status_code == 200, f'Self-rotate failed ({r.status_code}): {r.text}'

creds2 = r.json().get('credentials', {})
FINAL_CLIENT_ID     = creds2.get('clientId')
FINAL_CLIENT_SECRET = creds2.get('clientSecret')
print(f'✅ {PRINCIPAL_NAME} self-rotated successfully:')
print(f'   clientId:     {FINAL_CLIENT_ID}')
print(f'   clientSecret: {FINAL_CLIENT_SECRET}')
print('\n⚠️  Save these — this is the credential pair to actually use going forward.')

In [ ]:
# ============================================================
# Step 8 — VERIFY the final (self-rotated) credentials work end-to-end
# ============================================================
r = pc.get_token(FINAL_CLIENT_ID, FINAL_CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{PRINCIPAL_ROLE}')
assert r.status_code == 200, f'Final credentials do not work: {r.status_code} {r.text}'
print(f'✅ Final credentials authenticate successfully as {PRINCIPAL_NAME}')

# ⚠️ MEASURED LIVE (2026-07-08): the Step 5 (/reset) credentials do NOT die
# here. Polaris stores two secret slots per principal (main + secondary) and
# a plain /rotate only demotes the previous main secret to secondary — it
# stays valid. Full eviction takes a SECOND rotation (see Step 9). So the
# correct assertion at this point is that the Step-5 credentials STILL work:
r_step5 = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{PRINCIPAL_ROLE}')
assert r_step5.status_code == 200, (
    "Expected the Step-5 (/reset) credentials to still authenticate as the "
    "'secondary' secret after a single subsequent /rotate — got "
    f"{r_step5.status_code}. Has Polaris's rotation model changed?"
)
print(f"✅ Step-5 credentials still work ({r_step5.status_code}) — they're now the "
      "'secondary' secret, by design. This is NOT a leftover bug; see Step 9.")

In [ ]:
# ============================================================
# Step 9 — Prove full eviction requires a SECOND rotation
# ============================================================
# One /rotate call only shifts main→secondary (old secret survives one more
# generation). A second /rotate finally pushes the Step-5 credentials out of
# both slots. This cell rotates once more (using the CURRENT — Step 7 —
# credentials) and re-checks the Step-5 credentials afterward.
r = pc.get_token(FINAL_CLIENT_ID, FINAL_CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{PRINCIPAL_ROLE}')
assert r.status_code == 200
current_token = r.json()['access_token']

r = pc.rotate_credentials(PRINCIPAL_NAME, token=current_token)
assert r.status_code == 200, f'Second rotate failed: {r.status_code} {r.text}'
creds3 = r.json()['credentials']
print(f"✅ rotated again: clientId={creds3['clientId']}")

r_step5_final = pc.get_token(CLIENT_ID, CLIENT_SECRET, scope=f'PRINCIPAL_ROLE:{PRINCIPAL_ROLE}')
assert r_step5_final.status_code != 200, (
    f'Step-5 credentials STILL work after two rotations ({r_step5_final.status_code}) — '
    'expected full eviction by now.'
)
print(f'✅ Step-5 credentials are finally dead ({r_step5_final.status_code}) — took exactly '
      'two rotations past their issuance, as Polaris\'s main/secondary secret model predicts.')

print(f'\n✅ DONE — {PRINCIPAL_NAME} rotated end-to-end, and the two-rotation eviction '
      'rule is confirmed live, not just asserted from source-reading.')